# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [35]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
#if importlib.util.find_spec("pysqlite3") is not None:
#    import pysqlite3
#    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [36]:
# DONE: Import the necessary libs
# For example: 
import os
import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from lib.agents import Agent
from lib.llm import LLM
from lib.messages import UserMessage, SystemMessage, ToolMessage, AIMessage
from lib.tooling import tool
from lib.evaluation import EvaluationReport

In [37]:
# DONE: Load environment variables
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [38]:
# DONE: Create retrieve_game tool
# It should use chroma client and collection you created
chroma_client = chromadb.PersistentClient(path="chromadb")
collection = chroma_client.get_collection("udaplay")
@tool
def retrieve_game(query: str) -> list:
    """ Semantic search: Finds most results in the vector DB
        args:
            - query: a question about game industry. 
        You'll receive results as list. Each element contains:
            - Platform: like Game Boy, Playstation 5, Xbox 360...)
            - Name: Name of the Game
            - YearOfRelease: Year when that game was released for that platform
            - Description: Additional details about the game """
    results = collection.query(query_texts=[query], n_results=5)
    return [
        {
            "Name": meta["Name"],
            "Platform": meta["Platform"],
            "YearOfRelease": meta["YearOfRelease"],
            "Description": meta["Description"],
            "Publisher": meta["Publisher"],
        }
        for meta in results["metadatas"][0]
    ]

llm = LLM(api_key=OPENAI_API_KEY, tools=[retrieve_game])

# Quick test before building the full agent
result = retrieve_game("When was Pokémon Gold and Silver released?")
print(result)

[{'Name': 'Pokémon Gold and Silver', 'Platform': 'Game Boy Color', 'YearOfRelease': 1999, 'Description': 'Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.', 'Publisher': 'Nintendo'}, {'Name': 'Pokémon Ruby and Sapphire', 'Platform': 'Game Boy Advance', 'YearOfRelease': 2002, 'Description': 'Third-generation Pokémon games set in the Hoenn region, featuring new Pokémon and double battles.', 'Publisher': 'Nintendo'}, {'Name': 'Super Mario 64', 'Platform': 'Nintendo 64', 'YearOfRelease': 1996, 'Description': "A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.", 'Publisher': 'Nintendo'}, {'Name': 'Super Mario World', 'Platform': 'Super Nintendo Entertainment System (SNES)', 'YearOfRelease': 1990, 'Description': 'A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.', 'Publisher': 'Nintendo'}, {'Name': 'Mario Kart 8 Deluxe', 'Pla

#### Evaluate Retrieval Tool

In [39]:
# DONE: Create evaluate_retrieval tool
# You might use an LLM as judge in this tool to evaluate the performance
# You need to prompt that LLM with something like:
# "Your task is to evaluate if the documents are enough to respond the query. "
# "Give a detailed explanation, so it's possible to take an action to accept it or not."
# Use EvaluationReport to parse the result
# Tool Docstring:
#    Based on the user's question and on the list of retrieved documents, 
#    it will analyze the usability of the documents to respond to that question. 
#    args: 
#    - question: original question from user
#    - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
#    The result includes:
#    - useful: whether the documents are useful to answer the question
#    - description: description about the evaluation result

from lib.llm import LLM
from lib.parsers import PydanticOutputParser

judge_llm = LLM(api_key=OPENAI_API_KEY)

@tool
def evaluate_retrieval(question: str, retrieved_docs: list) -> dict:
    """Based on the user's question and on the list of retrieved documents,
    it will analyze the usability of the documents to respond to that question.
    args:
        - question: original question from user
        - retrieved_docs: retrieved documents most similar to the user query in the Vector Database
    The result includes:
        - useful: whether the documents are useful to answer the question
        - description: description about the evaluation result"""
    prompt = (
        "Your task is to evaluate if the documents are enough to respond the query. "
        f"\n# Question: {question}"
        f"\n# Documents: {retrieved_docs}"
        "\nDetermine if these documents contain enough information to answer the question."
    )
    response = judge_llm.invoke(prompt, response_format=EvaluationReport)
    parser = PydanticOutputParser(model_class=EvaluationReport)
    result = parser.parse(response)
    return {"useful": result.useful, "description": result.description}

# quick test
#docs = retrieve_game("When was Pokémon Gold and Silver released?")
#result = evaluate_retrieval("When was Pokémon Gold and Silver released?", docs)
#print(result)

#### Game Web Search Tool

In [40]:
# DONE: Create game_web_search tool
# Please use Tavily client to search the web
# Tool Docstring:
#    Semantic search: Finds most results in the vector DB
#    args:
#    - question: a question about game industry. 
from tavily import TavilyClient
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)

@tool
def game_web_search(question: str) -> list:
    """ Searches the web for game industry information.
    args:
        - question: a question about the game industry.
    Returns a list of search results with title, url, and content."""
    response = tavily_client.search(query=question, search_depth="basic")
    return [
        {
            "title": r["title"],
            "url": r["url"],
            "content": r["content"],
        }
        for r in response["results"]
    ]

# Quick test
result = game_web_search("What is Rockstar Games working on right now?")
print(result)

[{'title': "What will be Rockstar Games' next project after Grand Theft Auto VI?", 'url': 'https://www.facebook.com/groups/669950193844480/posts/1883724742467013', 'content': 'Rockstar Games has officially confirmed that Grand Theft Auto 6 is in active development. There have been a number of rumors regarding GTA 6'}, {'title': 'Rockstar games launcher seems to be down for me. Anyone else?', 'url': 'https://www.reddit.com/r/gtaonline/comments/1tle5ci/rockstar_games_launcher_seems_to_be_down_for_me', 'content': 'yep, me too. the launcher (PC) currently says: "Sorry, Rockstar Games is currently experiencing technical difficulties and online services'}, {'title': 'Rockstar Games down? Current problems and outages - US', 'url': 'https://downdetector.com/status/rockstar-games', 'content': 'User reports show no current problems with Rockstar Games ... Rockstar Games is the publisher of popular online games like Grand Theft Auto and Red Dead'}, {'title': 'Rockstar Games - Wikipedia', 'url': '

### Agent

In [51]:
# DONE: Create your Agent abstraction using StateMachine
# Equip with an appropriate model
# Craft a good set of instructions 
# Plug all Tools you developed

tools = [retrieve_game, evaluate_retrieval, game_web_search]
instructions = (
    "You are UdaPlay, an AI research assistant for the video game industry. "
    "Your workflow is: "
    "1. First, use retrieve_game to search the vector database for relevant game information. "
    "2. Then, use evaluate_retrieval to assess if the retrieved docs are sufficient. "
    "3. If not sufficient, use game_web_search to find information online. "
    "Provide clear, structured answers with citations where possible."
)

agent = Agent(
    model_name="gpt-4o-mini",
    instructions=instructions,
    tools=tools,
)

#run = agent.invoke("When was Pokémon Gold and Silver released?")
# final_state = run.get_final_state()
# answer = final_state["messages"][-1].content
#print(f"answer: {answer}")

In [ ]:
# DONE: Invoke your agent
# - When Pokémon Gold and Silver was released?
# - Which one was the first 3D platformer Mario game?
# - Was Mortal Kombat X realeased for Playstation 5?

def show_run(query, run):
    final = run.get_final_state()
    print(f"Query: {query}")
    print(f"Runs: {run.metadata['run_id']}")
    for msg in final["messages"]:
        role = msg.role
        if role == "assistant" and msg.content:
            print(f"\nAnswer: {msg.content}")
        elif role == "assistant" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"  [Tool called: {tc.function.name}({tc.function.arguments})]")
        elif role == "tool":
            print(f"  [Tool result: {msg.content[:100]}...]")
    print(f"Total tokens: {final.get('total_tokens', 'N/A')}")
    print("-" * 60)

query_1 = "When was Pokémon Gold and Silver released?"
run_1 = agent.invoke(query_1,session_id="q1")

query_2 = "Which one was the first 3D platformer Mario game?"
run_2 = agent.invoke(query_2,session_id="q2")

query_3 = "Was Mortal Kombat X released for PlayStation 5?"
run_3 = agent.invoke(query_3,session_id="q3")

show_run(query_1, run_1)
show_run(query_2, run_2)
show_run(query_3, run_3)

[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Terminating: __termination__
[StateMachine] Starting: __entry__
[StateMachine] Executing step: message_prep
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor
[StateMachine] Executing step: tool_executor
[StateMachine] Executing step: llm_processor


### (Optional) Advanced

In [43]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes